# BigQuery Description Auto-Generator with Gemini (Vertex AI)

이 노트북은 BigQuery 데이터셋을 스캔하여 Description(설명)이 누락된 테이블과 컬럼을 찾아내고, Google Cloud Vertex AI (Gemini)를 사용하여 자동으로 설명을 생성 및 업데이트하는 파이프라인입니다.

### **주요 기능**
1.  **Discovery**: `description`이 비어있는 테이블 및 컬럼 식별
2.  **Generation**: Gemini Pro 모델을 사용하여 메타데이터(테이블명, 컬럼명, 타입) 기반 설명 생성
3.  **Apply**: BigQuery 스키마 업데이트 (Native Dry Run 모드 지원)

### **사전 요구사항**
- Google Cloud Project 권한 (BigQuery User, Vertex AI User)
- 로컬 환경인 경우 `gcloud auth application-default login` 인증 필요
- Vertex AI API 사용 설정됨

In [ ]:
# 필요한 라이브러리 설치
!pip install --upgrade google-genai google-cloud-bigquery pandas db-dtypes

In [ ]:
import time
from typing import List, Dict, Any
from google import genai
from google.genai import types
from google.cloud import bigquery
import pandas as pd

# ==========================================
# [설정] 프로젝트 및 데이터셋 정보 입력
# ==========================================
PROJECT_ID = "your-project-id"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
DATASET_ID = "your_dataset_id"  # @param {type: "string", placeholder: "[your-dataset-id]", isTemplate: true}
LOCATION = "us-central1"        # Vertex AI location
MODEL_ID = "gemini-2.5-flash"   # Generation 에 사용할 모델

# True로 설정하면 실제 업데이트 대신 BigQuery의 Dry Run 기능을 사용하여 예상 결과를 확인합니다.
DRY_RUN = True  

# 클라이언트 초기화
bq_client = bigquery.Client(project=PROJECT_ID)
ai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

## 1. Gemini 프롬프트 및 생성 함수 정의

테이블 이름, 컬럼 이름, 데이터 타입을 입력받아 적절한 테이블 및 컬럼 설명을 각각 독립적으로 생성합니다.

In [ ]:
def get_table_sample_data(project_id: str, dataset_id: str, table_id: str, num_rows: int = 5) -> str:
    """
    테이블의 상위 N개 데이터를 조회하여 문자열 형식으로 반환합니다.
    실행 전 Dry Run을 통해 예상 바이트량을 출력합니다.
    """
    query = f"SELECT * FROM `{project_id}.{dataset_id}.{table_id}` LIMIT {num_rows}"
    try:
        # Native Dry Run 으로 예상 비용 확인
        dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
        dry_run_job = bq_client.query(query, job_config=dry_run_config)
        print(f"  [Info] Sample query for {table_id} will process {dry_run_job.total_bytes_processed} bytes.")

        # 실제 데이터 조회
        df_sample = bq_client.query(query).to_dataframe()
        if df_sample.empty:
            return "(데이터 없음)"
        return df_sample.to_string(index=False)
    except Exception as e:
        print(f"Error fetching sample data for {table_id}: {e}")
        return "(샘플 데이터를 가져올 수 없습니다)"

def generate_table_description(table_name: str, sample_data: str, context: str = "") -> str:
    """
    Gemini를 사용하여 테이블 설명을 생성합니다. (샘플 데이터 참고)
    """
    prompt_text = f"""
    당신은 숙련된 데이터 엔지니어입니다. 
    다음 정보와 샘플 데이터를 바탕으로 BigQuery 테이블에 대한 명확하고 간결한 영어로 된 description을 작성해주세요.
    
    [정보]
    - 테이블 이름: {table_name}
{f'- 추가 컨텍스트: {context}' if context else ''}
    
    [샘플 데이터 (일부)]
    {sample_data}
    
    [요구사항]
    - 100자 이내로 작성할 것.
    - 실제 데이터의 패턴과 의미를 반영하여 비즈니스 용어 위주로 작성할 것.
    - 오직 설명 텍스트만 출력할 것 (따옴표나 부가 설명 제외).
    """

    try:
        response = ai_client.models.generate_content(
            model=MODEL_ID,
            contents=[prompt_text],
            config=types.GenerateContentConfig(
                temperature=0.2, 
            )
        )
        return response.text.strip()
    except Exception as e:
        print(f"Error generating description for table {table_name}: {e}")
        return "테이블 설명 생성 실패"

def generate_column_description(table_name: str, column_name: str, data_type: str, sample_data: str, context: str = "") -> str:
    """
    Gemini를 사용하여 컬럼 설명을 생성합니다. (샘플 데이터 참고)
    """
    prompt_text = f"""
    당신은 숙련된 데이터 엔지니어입니다. 
    다음 정보와 샘플 데이터를 바탕으로 BigQuery 컬럼에 대한 명확하고 간결한 영어로 된 설명(Description)을 작성해주세요.
    
    [정보]
    - 테이블 이름: {table_name}
    - 컬럼 이름: {column_name}
    - 데이터 타입: {data_type}
{f'- 추가 컨텍스트: {context}' if context else ''}
    
    [샘플 데이터 (해당 테이블 일부)]
    {sample_data}
    
    [요구사항]
    - 50자 이내로 작성할 것.
    - 해당 컬럼의 데이터 값들을 참고하여 비즈니스 의미를 정확히 파악해 작성할 것.
    - 오직 설명 텍스트만 출력할 것 (따옴표나 부가 설명 제외).
    """

    try:
        response = ai_client.models.generate_content(
            model=MODEL_ID,
            contents=[prompt_text],
            config=types.GenerateContentConfig(
                temperature=0.2, 
            )
        )
        return response.text.strip()
    except Exception as e:
        print(f"Error generating description for column {column_name}: {e}")
        return "컬럼 설명 생성 실패"

## 2. BigQuery 스키마 탐색 및 업데이트 로직

데이터셋 내의 모든 테이블을 순회하며 누락된 테이블 설명과 컬럼 설명을 찾아서 차례대로 업데이트합니다.

In [ ]:
def escape_sql_string(s):
    """SQL 문자열 내의 단일 따옴표를 이스케이프 처리합니다."""
    return s.replace("'", "\\'")

def process_table_descriptions(dataset_id):
    dataset_ref = bq_client.dataset(dataset_id)
    tables = list(bq_client.list_tables(dataset_ref))
    
    print(f"Found {len(tables)} tables in dataset '{dataset_id}'")
    
    updates_log = []

    for table_item in tables:
        table_ref = dataset_ref.table(table_item.table_id)
        full_table = bq_client.get_table(table_ref)
        
        print(f"\n{'='*50}")
        print(f"Checking table: {full_table.table_id}...")

        # 1. 테이블 설명 처리
        if not full_table.description:
            print(f"  - Missing description for table: {full_table.table_id}")
            # 샘플 데이터 가져오기
            sample_data = get_table_sample_data(PROJECT_ID, dataset_id, full_table.table_id)
            
            generated_table_desc = generate_table_description(full_table.table_id, sample_data)
            print(f"    > Generated Table Desc: {generated_table_desc}")

            # SQL 문법 에러 방지를 위한 처리
            escaped_desc = escape_sql_string(generated_table_desc)
            
            updates_log.append({
                'table': full_table.table_id,
                'target': 'table',
                'name': full_table.table_id,
                'generated_description': generated_table_desc
            })

            # ALTER TABLE DDL 문 생성
            sql = f"ALTER TABLE `{PROJECT_ID}.{DATASET_ID}.{full_table.table_id}` SET OPTIONS(description='{escaped_desc}')"
            
            job_config = bigquery.QueryJobConfig(dry_run=DRY_RUN, use_query_cache=False)
            try:
                query_job = bq_client.query(sql, job_config=job_config)
                
                if DRY_RUN:
                    print(f"[DRY RUN] This query will process {query_job.total_bytes_processed} bytes.")
                    print(f"[DRY RUN] Validated DDL for {full_table.table_id}")
                else:
                    print(f"[SUCCESS] Updated description for {full_table.table_id}")
            except Exception as e:
                print(f"[ERROR] Failed to update {full_table.table_id}: {e}")
        else:
            print(f"  - Table already has description.")
            
    return pd.DataFrame(updates_log)


def process_column_descriptions(dataset_id):
    dataset_ref = bq_client.dataset(dataset_id)
    tables = list(bq_client.list_tables(dataset_ref))
    
    print(f"Found {len(tables)} tables in dataset '{dataset_id}'")
    
    updates_log = []

    for table_item in tables:
        table_ref = dataset_ref.table(table_item.table_id)
        full_table = bq_client.get_table(table_ref)
        
        print(f"\n{'='*50}")
        print(f"Checking table columns: {full_table.table_id}...")
        
        missing_desc_cols = [f for f in full_table.schema if not f.description]
        
        if not missing_desc_cols:
            print(f"  - All columns already have descriptions.")
            continue

        # 샘플 데이터 가져오기
        sample_data = get_table_sample_data(PROJECT_ID, dataset_id, full_table.table_id)
        
        for schema_field in full_table.schema:
            if not schema_field.description:
                print(f"  - Missing description for column: {schema_field.name} ({schema_field.field_type})")
                generated_col_desc = generate_column_description(
                    full_table.table_id, 
                    schema_field.name, 
                    schema_field.field_type,
                    sample_data
                )
                print(f"    > Generated Column Desc: {generated_col_desc}")
                
                escaped_desc = escape_sql_string(generated_col_desc)
                
                updates_log.append({
                    'table': full_table.table_id,
                    'target': 'column',
                    'name': schema_field.name,
                    'generated_description': generated_col_desc
                })

                # ALTER TABLE ALTER COLUMN DDL 문 생성
                sql = f"ALTER TABLE `{PROJECT_ID}.{DATASET_ID}.{full_table.table_id}` ALTER COLUMN `{schema_field.name}` SET OPTIONS(description='{escaped_desc}')"
                
                job_config = bigquery.QueryJobConfig(dry_run=DRY_RUN, use_query_cache=False)
                try:
                    query_job = bq_client.query(sql, job_config=job_config)
                    if DRY_RUN:
                        print(f"    [DRY RUN] This query will process {query_job.total_bytes_processed} bytes.")
                    else:
                        print(f"    [SUCCESS] Updated description for column {schema_field.name}")
                except Exception as e:
                    print(f"    [ERROR] Failed to update column {schema_field.name}: {e}")
            
    return pd.DataFrame(updates_log)

## 3. 실행

아래 셀을 실행하면 작업을 시작합니다. `DRY_RUN = True` 상태에서 결과를 확인한 후 False로 변경하여 적용하세요.

In [ ]:
# 실행 설정
RUN_TABLE_DESC = True   # 테이블 설명 업데이트 실행 여부
RUN_COLUMN_DESC = True  # 컬럼 설명 업데이트 실행 여부

all_updates = []

if RUN_TABLE_DESC:
    print("=== Starting Table Description Updates ===")
    df_table_updates = process_table_descriptions(DATASET_ID)
    if not df_table_updates.empty:
        all_updates.append(df_table_updates)
    print("=== Table Description Updates Completed ===\n")

if RUN_COLUMN_DESC:
    print("=== Starting Column Description Updates ===")
    df_column_updates = process_column_descriptions(DATASET_ID)
    if not df_column_updates.empty:
        all_updates.append(df_column_updates)
    print("=== Column Description Updates Completed ===\n")

# 결과 요약 출력
if all_updates:
    df_results = pd.concat(all_updates, ignore_index=True)
    print("\n[Summary of Updates]")
    display(df_results)
else:
    print("\n[Summary] No updates were made.")